# Projeto: Análise Estratégica da Superstore | SCTEC

## Análise Exploratória de Dados (EDA)

### 1. Objetivo do Projeto
Este projeto consiste na análise exploratória de dados (EDA) da "Superstore", utilizando o dataset disponibilizado no [Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final). 

Alinhado à crescente demanda por uma cultura *data-driven* no polo tecnológico da Grande Florianópolis, este estudo vai além da exploração técnica. O objetivo central é **transformar dados brutos em insights acionáveis**, permitindo a identificação de gargalos logísticos e oportunidades reais de otimização de rentabilidade. Através de uma visão analítica sobre métricas de *Supply Chain* e margens por segmento, buscaremos fundamentar a tomada de decisão estratégica, demonstrando que decisões baseadas em dados são o motor da eficiência operacional e do crescimento sustentável.

---

### 2. Perguntas de Negócio
Para guiar nossa análise, investigaremos as seguintes hipóteses:

* **Eficiência Logística:** Qual o impacto do *lead time* (tempo de entrega) na performance operacional?
* **Rentabilidade:** Quais categorias de produtos apresentam margens de lucro abaixo do esperado ou negativas?
* **Sazonalidade:** Existe um padrão sazonal nas vendas que impacta a gestão de estoque e o fluxo de caixa?

---

### 3. Metodologia da Análise Exploratória (EDA)
A nossa exploração será estruturada da seguinte forma:

* **Auditoria de Qualidade:** Identificação de valores ausentes, dados duplicados e inconsistências estruturais.
* **Análise Univariada:** Investigação da distribuição estatística das variáveis principais (lucro, vendas e descontos).
* **Análise de Correlação:** Investigação da sensibilidade do lucro em relação à estratégia de descontos aplicada.
* **Análise Temporal:** Avaliação do comportamento das vendas, lucro e desconto ao longo do tempo (identificação de sazonalidade).

## Importação de bibliotecas

In [1]:
import os
import pandas as pd
from pathlib import Path
from prophet import Prophet
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("✓ Ambiente configurado e bibliotecas importadas.")

/Users/wilken/Documents/jobs/2026/formacoes/SCTEC/desafio_sctec/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Ambiente configurado e bibliotecas importadas.


## Função para padronizar a geração de figuras 

In [2]:
import plotly.io as pio

# 1. Escolha um template mais limpo (o 'plotly_white' é excelente para relatórios)
pio.templates.default = "plotly_white"

# 2. Defina os padrões globais de layout
pio.templates["plotly_white"].layout.update(
    width=900,          # Largura padrão em pixels
    height=500,         # Altura padrão em pixels
    margin=dict(l=40, r=40, b=40, t=60), # Margens para o gráfico não ficar "encostado" nas bordas
    font=dict(family="Arial", size=12)   # Padronização de fonte
)

Layout({
    'annotationdefaults': {'arrowcolor': '#2a3f5f', 'arrowhead': 0, 'arrowwidth': 1},
    'autotypenumbers': 'strict',
    'coloraxis': {'colorbar': {'outlinewidth': 0, 'ticks': ''}},
    'colorscale': {'diverging': [[0, '#8e0152'], [0.1, '#c51b7d'], [0.2,
                                 '#de77ae'], [0.3, '#f1b6da'], [0.4, '#fde0ef'],
                                 [0.5, '#f7f7f7'], [0.6, '#e6f5d0'], [0.7,
                                 '#b8e186'], [0.8, '#7fbc41'], [0.9, '#4d9221'],
                                 [1, '#276419']],
                   'sequential': [[0.0, '#0d0887'], [0.1111111111111111,
                                  '#46039f'], [0.2222222222222222, '#7201a8'],
                                  [0.3333333333333333, '#9c179e'],
                                  [0.4444444444444444, '#bd3786'],
                                  [0.5555555555555556, '#d8576b'],
                                  [0.6666666666666666, '#ed7953'],
                           

## 🔎 1. Ingestão de Dados

In [3]:
# Definindo o caminho local usando pathlib (garante que funciona em Windows, Mac ou Linux)
# Aqui nos certificamos de que o dataset esteja em '/data'

file_path = Path('data/sample_superstore.csv')

try:
    # Carregamento com o encoding 'latin-1' (validado para este dataset)
    df = pd.read_csv(file_path, encoding='latin-1')
    print(f"✓ Dataset carregado com sucesso: {df.shape[0]} linhas, {df.shape[1]} colunas.")
except FileNotFoundError:
    print(f"❌ Erro: Arquivo não encontrado em {file_path}.")
    print("Dica: Verifique se a pasta 'data' existe e se o arquivo está lá.")

✓ Dataset carregado com sucesso: 9994 linhas, 21 colunas.


## 🔎 2. Auditoria de Integridade dos Dados (Sanity Check)

In [4]:
print("\n=== INFORMAÇÕES DO DATASET ===")
print(f"Forma: {df.shape}")
print(f"\nColunas:\n{df.dtypes}")
print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")
print(f"\nPrimeiras 5 linhas:")
df.head()


=== INFORMAÇÕES DO DATASET ===
Forma: (9994, 21)

Colunas:
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Valores nulos por coluna:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount     

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


O dataset foi carregado com **9.994 registros** e **zero valores nulos**, indicando uma alta qualidade estrutural inicial.

**Observações prática para as próximas etapas referente a tipagem de dados:**

As colunas `Order Date` e `Ship Date` estão como `str` (texto). A conversão para o formato `datetime` é o próximo passo necessário para habilitar a manipulação temporal (feature engineering) e viabilizar as análises de série temporal. Tal operação será realizada na seção 4.

## 🔎 3. Data Profiling (Estatísticas descritivas)

In [5]:
# Primeiro obtemos médias, medianas, minímos e máximos.

print(df.describe())

# Verificar valores únicos em colunas categóricas, isso será útil para o design das futuras análises.

print(f"\nCategorias: {df['Category'].unique()}")
print(f"Segmentos: {df['Segment'].unique()}")
print(f"Regiões: {df['Region'].unique()}")

            Row ID   Postal Code         Sales     Quantity     Discount  \
count  9994.000000   9994.000000   9994.000000  9994.000000  9994.000000   
mean   4997.500000  55190.379428    229.858001     3.789574     0.156203   
std    2885.163629  32063.693350    623.245101     2.225110     0.206452   
min       1.000000   1040.000000      0.444000     1.000000     0.000000   
25%    2499.250000  23223.000000     17.280000     2.000000     0.000000   
50%    4997.500000  56430.500000     54.490000     3.000000     0.200000   
75%    7495.750000  90008.000000    209.940000     5.000000     0.200000   
max    9994.000000  99301.000000  22638.480000    14.000000     0.800000   

            Profit  
count  9994.000000  
mean     28.656896  
std     234.260108  
min   -6599.978000  
25%       1.728750  
50%       8.666500  
75%      29.364000  
max    8399.976000  

Categorias: <StringArray>
['Furniture', 'Office Supplies', 'Technology']
Length: 3, dtype: str
Segmentos: <StringArray>
['Con

A partir de estatísticas descritivas (média, mediana, desvio padrão), o diagnóstico numérico revelou dois comportamentos críticos que guiarão nossas próximas análises:

- **Assimetria (Sales):** A discrepância entre a média ($229) e a mediana ($54) confirma uma distribuição de "cauda longa". A maioria das vendas é de baixo valor, com poucos pedidos outliers elevando a média.
- **Volatilidade (Profit):** O lucro apresenta instabilidade severa. O desvio padrão ($234) é quase 10x superior à média ($28), indicando a presença de operações com prejuízos profundos que precisam ser isoladas.

## 🔎 4. Pré-processamento de Dados e Engenharia de Features

In [6]:
# Pré processamento
# Converter datas para datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')
print("✓ Datas convertidas para datetime")

# Engenharia de features
# Criar colunas de tempo
df['Ano'] = df['Order Date'].dt.year
df['Mes'] = df['Order Date'].dt.month
df['Trimestre'] = df['Order Date'].dt.quarter
df['Data_Semana'] = (df['Ship Date'] - df['Order Date']).dt.days
print("✓ Colunas de tempo criadas (Ano, Mes, Trimestre, Data_Semana)")

# Verificar valores negativos em lucro (são válidos, empresas com prejuízo)
lucros_negativos = (df['Profit'] < 0).sum()
print(f"✓ Linhas com lucro negativo: {lucros_negativos} (mantidas, são dados válidos)")

# Verificar descontos acima de 100% (inválido)
descontos_invalidos = (df['Discount'] > 1).sum()
if descontos_invalidos > 0:
    print(f"⚠ Descontos > 100%: {descontos_invalidos} (podem ser removidos)")
else:
    print("✓ Todos os descontos estão no intervalo [0, 1]")

# Remover duplicatas completas (se houver)
duplicatas = df.duplicated().sum()
if duplicatas > 0:
    df = df.drop_duplicates()
    print(f"✓ {duplicatas} duplicatas removidas")
else:
    print("✓ Nenhuma duplicata encontrada")

print(f"\n✓ Dataset final: {df.shape[0]} linhas após limpeza")

✓ Datas convertidas para datetime
✓ Colunas de tempo criadas (Ano, Mes, Trimestre, Data_Semana)
✓ Linhas com lucro negativo: 1871 (mantidas, são dados válidos)
✓ Todos os descontos estão no intervalo [0, 1]
✓ Nenhuma duplicata encontrada

✓ Dataset final: 9994 linhas após limpeza


## 🏁

O pipeline de pré processamento de dados e engenharia de features foi executado com sucesso. O dataset encontra-se estruturado e validado para a fase de exploração. Esta etapa consolidou a preparação do dataset, garantindo que os dados estejam prontos para análises estatísticas e temporais robustas.

- **Integridade:** Tipos de dados convertidos (datetime) e ausência de registros duplicados confirmada.
- **Enriquecimento:** Criação de colunas temporais (`Ano`, `Mes`, `Trimestre`, `Data_Semana`) para viabilizar as análises temporais.
- **Validação de Negócio:** Confirmada a consistência das métricas financeiras. As 1.871 ocorrências de lucro negativo foram validadas como registros operacionais reais e mantidas para análise.
Além disso, validamos o intervalo de descontos (todos entre 0 e 1). Nenhum outlier ou valor inválido detectado.


## 📊 Análise 1: Distribuição regional de lucro

In [74]:
# Por trimestre
trimestre_sales = df.groupby(['Ano', 'Trimestre'])[['Sales', 'Profit']].sum().reset_index()
trimestre_sales['Periodo'] = trimestre_sales['Ano'].astype(str) + ' Q' + trimestre_sales['Trimestre'].astype(str)

print(trimestre_sales)

# Gráfico de linhas
fig_tendencia = go.Figure()
fig_tendencia.add_trace(go.Scatter(
    x=trimestre_sales['Periodo'],
    y=trimestre_sales['Sales'],
    mode='lines+markers',
    name='Vendas',
    line=dict(color='#636EFA', width=3),
    marker=dict(size=8)
))
fig_tendencia.add_trace(go.Scatter(
    x=trimestre_sales['Periodo'],
    y=trimestre_sales['Profit'],
    mode='lines+markers',
    name='Lucro',
    line=dict(color='#EF553B', width=3),
    marker=dict(size=8),
    yaxis='y2'
))

fig_tendencia.update_layout(
    title='Tendência de Vendas e Lucro por Trimestre',
    xaxis_title='Período',
    yaxis_title='Vendas ($)',
    yaxis2=dict(title='Lucro ($)', overlaying='y', side='right'),
    height=500,
    hovermode='x unified',
    template='plotly_white'
)
fig_tendencia.show()

fig.write_image("images/analise_6_vendas_ao_longo_do_tempo.png")

     Ano  Trimestre        Sales      Profit  Periodo
0   2014          1   74447.7960   3811.2290  2014 Q1
1   2014          2   86538.7596  11204.0692  2014 Q2
2   2014          3  143633.2123  12804.7218  2014 Q3
3   2014          4  179627.7302  21723.9541  2014 Q4
4   2015          1   68851.7386   9264.9416  2015 Q1
5   2015          2   89124.1870  12190.9224  2015 Q2
6   2015          3  130259.5752  16853.6194  2015 Q3
7   2015          4  182297.0082  23309.1203  2015 Q4
8   2016          1   93237.1810  11441.3708  2016 Q1
9   2016          2  136082.3010  16390.3394  2016 Q2
10  2016          3  143787.3622  15823.6048  2016 Q3
11  2016          4  236098.7538  38139.8593  2016 Q4
12  2017          1  123144.8602  23506.2026  2017 Q1
13  2017          2  133764.3720  15499.2085  2017 Q2
14  2017          3  196251.9560  26985.1325  2017 Q3
15  2017          4  280054.0670  27448.7260  2017 Q4


In [75]:
import plotly.subplots as sp
import plotly.graph_objects as go

# Agrupar os dados
data_region = df.groupby('Region')[['Sales', 'Profit']].sum().reset_index()

# Criar a estrutura de subplots (1 linha, 2 colunas)
fig = sp.make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=('Distribuição de Vendas', 'Distribuição de Lucro')
)

# Adicionar o gráfico de Vendas
fig.add_trace(go.Pie(
    labels=data_region['Region'], 
    values=data_region['Sales'], 
    hole=0.3, 
    name='Vendas'
), 1, 1)

# Adicionar o gráfico de Lucro
fig.add_trace(go.Pie(
    labels=data_region['Region'], 
    values=data_region['Profit'], 
    hole=0.3, 
    name='Lucro'
), 1, 2)

# Ajustes finais
fig.update_layout(title_text='Vendas vs Lucro por Região', height=500)
fig.show()

# Exportar
fig.write_image("images/analise_vendas_vs_lucro_regiao.png")

## 📊 Análise 2: Desempenho e rentabilidade por segmento

## 📊 Análise 3: Confirmação Visual da Distribuição

Validando a assimetria e a presença de *outliers* identificadas na estatística descritiva. O foco aqui é confirmar o comportamento de "cauda longa" nas Vendas e a instabilidade nas margens de Lucro.

In [9]:
fig = make_subplots(
    rows=2, cols=2, 
    subplot_titles=('Distribuição de Vendas (Geral)', 'Foco nas Vendas (Zoom < $2k)', 
                    'Distribuição de Lucro (Geral)', 'Foco no Lucro (Zoom entre -500 e 500)'),
    horizontal_spacing=0.15, vertical_spacing=0.15
)

# Template Limpo
template = "plotly_white"

# Vendas Geral
fig.add_trace(go.Histogram(x=df['Sales'], name='Vendas Geral', marker_color='#1f77b4'), row=1, col=1)
# Vendas Zoom
fig.add_trace(go.Histogram(x=df[df['Sales'] < 2000]['Sales'], name='Vendas Zoom', marker_color='#1f77b4'), row=1, col=2)

# Lucro Geral
fig.add_trace(go.Histogram(x=df['Profit'], name='Lucro Geral', marker_color='#d62728'), row=2, col=1)
# Lucro Zoom
fig.add_trace(go.Histogram(x=df[(df['Profit'] > -500) & (df['Profit'] < 500)]['Profit'], name='Lucro Zoom', marker_color='#d62728'), row=2, col=2)

# Iterando sobre as linhas (rows) e colunas (cols) para adicionar os nomes dos eixos.
for row in [1, 2]:
    for col in [1, 2]:
        # Lógica para definir o título do eixo X dinamicamente
        # Linha 1 = Vendas, Linha 2 = Lucro
        x_title = "Valor de Vendas ($)" if row == 1 else "Lucro ($)"
        
        fig.update_xaxes(title_text=x_title, row=row, col=col)
        fig.update_yaxes(title_text="Frequência", row=row, col=col)
        
fig.update_layout(height=800, showlegend=False, template=template, title_text="Distribuição de Vendas e Lucro Através de Histogramas")

# Salvar
fig.write_image("images/analise_3_vendas_lucro_histograma.png", scale=2)
fig.show()

In [10]:
# Calcular a assimetria original
skew_original = df['Profit'].skew()

# Remover os outliers (ex: maiores que 1000 - ajustamos conforme sua percepção)
df_sem_outliers = df[df['Profit'] < 1000]
skew_sem_outliers = df_sem_outliers['Profit'].skew()

print(f"Assimetria Original: {skew_original:.2f}")
print(f"Assimetria sem Outliers (>1k): {skew_sem_outliers:.2f}")

Assimetria Original: 7.56
Assimetria sem Outliers (>1k): -13.99


In [11]:
# Usando px.scatter com marginais para análise de outliers usando dispersão + distribuição
fig = px.scatter(df, 
                 x="Sales", 
                 y="Profit", 
                 marginal_x="histogram", 
                 marginal_y="histogram",
                 title="Identificação de Outliers (Vendas vs Lucro)",
                 template="plotly_white",
                 opacity=0.6)

# Atualizando títulos para ficar profissional
fig.update_layout(xaxis_title="Valor de Vendas ($)", yaxis_title="Lucro ($)")

# Salvar
fig.write_image("images/analise_3_outliers_scatterplot.png", scale=2)
fig.show()

Nesta análise, iniciamos com histogramas para mapear a distribuição de valor de vendas e lucro. Mas a limitação é a seguinte: histogramas simples "achatam" os dados, escondendo o comportamento da massa de transações devido aos *outliers* de alto valor.

Para enriquecermos a nossa análise aplicamos o cálculo de *Skewness/Assimetria* para quantificar essa distorção. Descobrimos que a assimetria positiva do total mascara uma estrutura cronicamente deficitária (assimetria negativa) na operação real quanto tiramos outliers do cálculo.

Para superar as limitações do histograma, utilizamos o *Joint Plot* na sequência. Esta representação permitiu visualizar não apenas a distribuição (cauda longa), mas a correlação real entre volume de vendas e prejuízo. Identificamos que o prejuízo não é um erro pontual, mas um comportamento que escala com certos tipos de transação.

## 📊 Análise 4: Impacto do desconto na rentabilidade usando regressão linear

In [12]:
correlacao = df['Discount'].corr(df['Profit'])
print(f"Correlação Desconto vs Lucro: {correlacao:.3f}")

# Estatísticas por faixa de desconto
df['Faixa_Desconto'] = pd.cut(df['Discount'],
                               bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0],
                               labels=['0%', '1-10%', '11-20%', '21-30%', '31-100%'])
faixa_stats = df.groupby('Faixa_Desconto')[['Sales', 'Profit']].agg(['sum', 'mean']).round(2)
print(f"\nEstatísticas por faixa de desconto:\n{faixa_stats}")

# Gráfico scatter com tendência
fig_desconto = px.scatter(
    df.sample(min(500, len(df))),  # Amostra pra não sobrecarregar
    x='Discount',
    y='Profit',
    trendline='ols',
    title='Desconto vs Lucro (com linha de tendência)',
    labels={'Discount': 'Taxa de Desconto', 'Profit': 'Lucro ($)'},
    opacity=0.5,
    template='plotly_white'
)
fig_desconto.update_layout(height=500)
fig_desconto.write_image("images/analise_4_desconto_lucro.png", scale=2)
fig_desconto.show()


Correlação Desconto vs Lucro: -0.219

Estatísticas por faixa de desconto:
                    Sales            Profit        
                      sum    mean       sum    mean
Faixa_Desconto                                     
0%               54369.35  578.40   9029.18   96.06
1-10%           792152.89  213.58  91756.30   24.74
11-20%          103226.66  454.74 -10369.28  -45.68
21-30%          195314.76  630.05 -48447.73 -156.28
31-100%          64228.74   75.03 -76559.05  -89.44


In [72]:
# Vamos criar um gráfico de dispersão para identificar a "Zona de Morte" do lucro
import plotly.express as px

# Criando um gráfico que mostra a relação entre Desconto e Lucro
fig_elasticidade = px.scatter(
    df, 
    x='Discount', 
    y='Profit', 
    color='Profit', 
    size='Sales',
    title='Relação: Desconto vs. Lucro (Identificando a Zona de Risco)',
    color_continuous_scale='RdYlGn' # Vermelho (prejuízo) para Verde (lucro)
)

fig_elasticidade.update_layout(title_x=0.5, template='plotly_white')
fig_elasticidade.show()

In [13]:
# Analisando a correlação entre Discount e Profit
correlation = df[['Discount', 'Profit']].corr().iloc[0, 1]
print(f"Coeficiente de Correlação (Discount vs Profit): {correlation:.3f}")

Coeficiente de Correlação (Discount vs Profit): -0.219


A análise de dispersão revela uma correlação negativa entre Desconto e Lucro. O modelo demonstra alta sensibilidade à variável de desconto: o potencial de lucro alto é exclusivo de transações com 0% de desconto. Observamos um 'ponto de inversão' claro: a partir de 11% de desconto, a média de lucro torna-se negativa, e acima de 30%, o prejuízo torna-se um comportamento estrutural, com ausência de qualquer transação rentável.

Além disso, o cálculo estatístico da correlação global entre Desconto e Lucro resultou em um coeficiente de -0.219 (natureza fraca), tornando esse indicador insuficiente para explicar a variabilidade do lucro no modelo. Portanto, não trataremos o desconto como uma dependência linear direta; a política de precificação deve ser avaliada sob uma ótica qualitativa e estratégica, priorizando o comportamento por categoria."

## 📊 Análise 5: Vendas e lucro por categoria

In [76]:
# 1. Agrupando os dados por Segmento
df_segment = df.groupby('Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()

# 2. Calculando a Margem de Lucro (o indicador real de rentabilidade)
df_segment['Profit Margin'] = (df_segment['Profit'] / df_segment['Sales']) * 100

# 3. Visualização: Performance (Sales) vs Rentabilidade (Margin)
fig = px.scatter(df_segment, 
                 x='Sales', 
                 y='Profit Margin', 
                 size='Profit', 
                 color='Segment',
                 title='Performance (Volume de Vendas) vs. Rentabilidade (Margem %)',
                 labels={'Sales': 'Volume de Vendas (Total)', 'Profit Margin': 'Margem de Lucro (%)'},
                 template='plotly_white')

fig.show()

fig.write_image("images/analise_2_desempenho_rentabilidade_segmento.png")

In [14]:
cat_sales = df.groupby('Category')[['Sales', 'Profit']].agg(['sum', 'mean', 'count']).round(2)
print(cat_sales)

fig_categoria = px.bar(
    df.groupby('Category')[['Sales', 'Profit']].sum().reset_index(),
    x='Category',
    y=['Sales', 'Profit'],
    barmode='group',
    title='Vendas e Lucro Total por Categoria',
    labels={'value': 'Valor ($)', 'Category': 'Categoria'},
    color_discrete_map={'Sales': '#636EFA', 'Profit': '#EF553B'},
    template='plotly_white'
)

fig_categoria.update_layout(height=500, showlegend=True)
fig_categoria.show()

fig.write_image("images/analise_5_venda_lucro_por_categorias.png")

                     Sales                   Profit             
                       sum    mean count        sum   mean count
Category                                                        
Furniture        741999.80  349.83  2121   18451.27   8.70  2121
Office Supplies  719047.03  119.32  6026  122490.80  20.33  6026
Technology       836154.03  452.71  1847  145454.95  78.75  1847


A categoria de Furniture (Móveis) apresenta um alto volume de vendas, mas uma rentabilidade desproporcionalmente baixa em comparação às demais. Isso sugere que a categoria pode estar operando sob uma política de descontos mais agressiva, que corrói suas margens de lucro. Utilizaremos um boxplot para comparar a distribuição e a variabilidade das taxas de desconto aplicadas em cada categoria. Essa análise confirmará se a estratégia de precificação é a causa raiz do desempenho inferior desta categoria.

In [15]:
# Criando o boxplot com visualização de pontos
fig = px.box(df, 
             x='Category', 
             y='Discount', 
             color='Category',
             #points='all',         # <--- Isso coloca os pontos individuais no gráfico
             notched=False,        # Deixa a caixa reta e mais limpa
             title='Distribuição da Taxa de Desconto por Categoria',
             template='plotly_white')

# Ajustes finos para deixar o gráfico mais "leve"
fig.update_layout(
    xaxis_title='Categoria',
    yaxis_title='Taxa de Desconto',
    showlegend=False,
    boxmode='group' # Garante que as caixas fiquem organizadas lado a lado
)

# Deixa as caixas um pouco mais transparentes para ver melhor os pontos
fig.update_traces(opacity=0.6) 

fig.show()

fig.write_image("images/analise_5_taxa_de_desconto_por_categorias_boxplot.png")

O boxplot confirmou que o prejuízo em Móveis decorre de uma política de descontos despadronizada e agressiva. Enquanto outras categorias operam com taxas constantes e previsíveis, a dispersão em Móveis sugere uma gestão de preços menos eficiente.

## 📊 Análise 6: Tendências de vendas e descontos ao longo do tempo

In [17]:
# Convertendo data para datetime se necessário
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Criando um dataframe com a média de desconto por trimestre e categoria
df_temporal = df.groupby([df['Order Date'].dt.to_period('Q'), 'Category'])['Discount'].mean().reset_index()
df_temporal['Order Date'] = df_temporal['Order Date'].dt.to_timestamp()

# Plotando
fig = px.line(df_temporal, 
              x='Order Date', 
              y='Discount', 
              color='Category', 
              title='Evolução da Média de Descontos por Categoria ao longo do tempo',
              markers=True,
              template='plotly_white')

fig.show()

fig.write_image("images/analise_6_decontos_longo_do_tempo.png")

A análise temporal demonstra que, apesar da sazonalidade comum entre vendas e lucro, as estratégias de precificação das categorias operam de forma distinta. Enquanto Technology mantém margens sólidas com descontos conservadores, Furniture aplica descontos agressivos que não se revertem em faturamento proporcional. Essa disparidade evidencia um desequilíbrio na eficiência operacional, onde o custo dos descontos em Furniture não impulsiona o crescimento esperado para a categoria.

## 📊 Análise 7: Top subcategorias

In [18]:
top_subcat = df.groupby('Sub-Category')[['Sales', 'Profit', 'Quantity']].agg(['sum', 'mean']).round(2)
top_subcat_sorted = top_subcat.sort_values(('Profit', 'sum'), ascending=False).head(10)
print(top_subcat_sorted)

# Gráfico horizontal
fig_subcat = px.bar(
    df.groupby('Sub-Category')['Profit'].sum().sort_values().tail(10).reset_index(),
    x='Profit',
    y='Sub-Category',
    orientation='h',
    title='Top 10 Subcategorias por Lucro Total',
    labels={'Profit': 'Lucro ($)', 'Sub-Category': 'Subcategoria'},
    color='Profit',
    color_continuous_scale='Viridis'
)
fig_subcat.update_layout(height=500)
fig_subcat.update_traces(texttemplate='%{x:$.2s}', textposition='outside')
fig_subcat.show()


fig.write_image("images/analise_7_top_subcategorias.png")

                  Sales             Profit         Quantity      
                    sum     mean       sum    mean      sum  mean
Sub-Category                                                     
Copiers       149528.03  2198.94  55617.82  817.91      234  3.44
Phones        330007.05   371.21  44515.73   50.07     3289  3.70
Accessories   167380.32   215.97  41936.64   54.11     2976  3.84
Paper          78479.21    57.28  34053.57   24.86     5178  3.78
Binders       203412.73   133.56  30221.76   19.84     5974  3.92
Chairs        328449.10   532.33  26590.17   43.10     2356  3.82
Storage       223843.61   264.59  21278.83   25.15     3158  3.73
Appliances    107532.16   230.76  18138.01   38.92     1729  3.71
Furnishings    91705.16    95.83  13059.14   13.65     3563  3.72
Envelopes      16476.40    64.87   6964.18   27.42      906  3.57


## 🏁 Conclusão da Análise Exploratória (EDA)

Após nossa investigação detalhada, os pilares da operação da Superstore estão claros:

* **Sazonalidade:** Crescimento concentrado em Q4, mas que mascara ineficiências operacionais.
* **Paradoxo do Volume:** O segmento *Consumer* é o nosso maior volume, mas o menos rentável.
* **O Vilão da Margem:** A categoria *Furniture* opera com descontos agressivos que corroem a rentabilidade (ponto de inversão em 11%).
* **Subcategorias:** *Copiers* são o nosso "motor de valor", enquanto *Phones* e *Binders* atuam como "motores de volume" com margens comprimidas.

Com base nesse diagnóstico, estamos prontos para a etapa final do nosso estudo, a **Fase de Modelagem**, onde utilizaremos Machine Learning para realizar previsões de performance e clusterização de comportamento, validando nossas teses de negócio com rigor estatístico.

## 🧑🏾‍💻 Modelagem a partir de modelos de ML

**Objetivos:** 

1. Prever a tendência orgânica de faturamento para os próximos 6 meses e validar a sazonalidade do negócio.
2. Categorização e clusterização
3. Otimização

In [68]:
# Primeiro criamos uma cópia para modelagem a seguir para não alterar o dataset original de EDA
df_ml = df.copy()

# Exibindo as primeiras linhas para checagem final antes do ML
print("✓ Dataset preparado para modelagem.")
#display(df_ml.head())

# 1. Garantir que a coluna de data esteja no formato correto
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Criando o dataset de Série Temporal (df_ts)
# Vamos agrupar por mês ('M') e somar Sales e Profit
df_ts = df.set_index('Order Date').resample('ME')[['Sales', 'Profit']].sum().reset_index()

checking = False

if checking:

    # Visualização
    fig = px.line(df_ts, x='Order Date', y='Sales', title='Tendência Mensal de Vendas')
    fig.update_layout(title_x=0.5, template='plotly_white')
    fig.show()

    # Exibir as primeiras linhas da nossa nova estrutura
    print("Estrutura do dataset para previsão:")
    display(df_ts.head())

✓ Dataset preparado para modelagem.


## 🔮 Modelagem 1: Forecasting de Vendas (Série Temporal com Prophet)

**Objetivo:** Prever a tendência orgânica de faturamento para os próximos 6 meses e validar a sazonalidade do negócio.

### O Modelo Prophet

O Prophet é um modelo de ML desenvolvido pelo time de Ciência de Dados do Facebook para previsão de séries temporais. Ele decompõe dados históricos em três componentes: tendência (padrão geral de crescimento ou queda), sazonalidade (ciclos repetitivos) e efeitos de feriados. É especialmente eficaz para negócios com ciclos sazonais pronunciados, como observamos nos dados da Superstore.

### Metodologia de Validação

Adotamos uma abordagem profissional em duas etapas:

1. **Etapa de Validação:** Treinamos o modelo com 80% dos dados históricos (38 meses) e testamos em 20% (10 meses) que o modelo nunca viu. Isso garante que as métricas de erro refletem o desempenho real.

2. **Etapa de Produção:** Após validar que o modelo é confiável, treinamos uma nova instância com 100% dos dados históricos para gerar a previsão futura, maximizando a informação disponível.

### Métricas de Avaliação

Para quantificar a precisão do modelo, utilizamos três métricas complementares:

**MAE** (Mean Absolute Error): Mede o erro médio em dólares, quantificando o desvio absoluto esperado por mês.

**RMSE** (Root Mean Squared Error): Penaliza desvios extremos, garantindo sensibilidade a picos sazonais críticos para gestão de estoque.

**MAPE** (Mean Absolute Percentage Error): Expressa a acurácia em percentual, oferecendo a perspectiva mais clara para decisões executivas.

In [60]:
# Preparar os dados
df_prophet = df_ts.rename(columns={'Order Date': 'ds', 'Sales': 'y'}).copy()

# ---------------------------------------------------------
# ETAPA 1: VALIDAÇÃO (Backtesting)
# ---------------------------------------------------------
split_point = int(len(df_prophet) * 0.8)
df_train = df_prophet[:split_point].copy()
df_test = df_prophet[split_point:].copy()

# Criamos o modelo de validação
model_val = Prophet(yearly_seasonality=True, daily_seasonality=False, interval_width=0.95)
model_val.fit(df_train)

# Validamos nos dados de teste (os 20%)
forecast_val = model_val.predict(df_test[['ds']])
y_pred_val = forecast_val['yhat'].values
y_true_val = df_test['y'].values

# Métricas da Validação
rmse = np.sqrt(mean_squared_error(y_true_val, y_pred_val))
mae = mean_absolute_error(y_true_val, y_pred_val) # Calculando o MAE agora!
mape = np.mean(np.abs((y_true_val - y_pred_val) / y_true_val)) * 100

print(f"🎯 PERFORMANCE DO MODELO (Validação 80/20):")
print(f"MAPE: {mape:.2f}% | RMSE: ${rmse:,.2f} | MAE: ${mae:,.2f}")

# A partir daqui, você decide: se o erro for aceitável, seguimos para o modelo final.

14:14:05 - cmdstanpy - INFO - Chain [1] start processing
14:14:05 - cmdstanpy - INFO - Chain [1] done processing


🎯 PERFORMANCE DO MODELO (Validação 80/20):
MAPE: 16.15% | RMSE: $14,811.52 | MAE: $11,286.44


In [ ]:
# ---------------------------------------------------------
# ETAPA 2: VISUALIZAÇÃO
# ---------------------------------------------------------
fig_forecast = go.Figure()

# ===== DADOS REAIS =====
# Combina treino + teste (tudo o que realmente aconteceu)
df_real = pd.concat([df_train, df_test], ignore_index=False)

fig_forecast.add_trace(go.Scatter(
    x=df_real['ds'], y=df_real['y'], 
    mode='lines', name='Dados Reais', 
    line=dict(color='#185FA5', width=2.5)
))

# ===== PREVISÕES =====
# Teste previsto (o que o modelo acha que foi)
fig_forecast.add_trace(go.Scatter(
    x=forecast_test['ds'], y=forecast_test['yhat'], 
    mode='lines', name='Modelagem (Período de Teste)', 
    line=dict(color='#185FA5', dash='dash', width=2)
))

# Previsão futura (o que o modelo acha que será)
fig_forecast.add_trace(go.Scatter(
    x=future_only['ds'], y=future_only['yhat'], 
    mode='lines', name='Modelagem (Próximos 6 meses)', 
    line=dict(color='#A32D2D', dash='dash', width=3)
))

# Intervalo de confiança superior
fig_forecast.add_trace(go.Scatter(
    x=future_only['ds'], y=future_only['yhat_upper'],
    fill=None, mode='lines',
    line_color='rgba(0,0,0,0)',
    showlegend=False
))

# Intervalo de confiança (banda preenchida)
fig_forecast.add_trace(go.Scatter(
    x=future_only['ds'], y=future_only['yhat_lower'],
    fill='tonexty', mode='lines',
    line_color='rgba(0,0,0,0)',
    name='Intervalo de Confiança (95%)',
    fillcolor='rgba(163, 45, 45, 0.15)'
))

fig_forecast.update_layout(
    title={
        'text': 'Forecasting de Vendas: Modelo Prophet',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#2C2C2A', 'family': 'Arial, sans-serif'}
    },
    xaxis_title='Período (Mensal)',
    yaxis_title='Vendas (USD)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.98,
        xanchor='left',
        x=0.02,
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='#888780',
        borderwidth=0.5,
        font=dict(size=11)
    ),
    height=650,
    width=1000,
    font=dict(family='Arial, sans-serif', size=12),
    plot_bgcolor='white', #'rgba(241, 239, 232, 0.3)',
    paper_bgcolor='white',
    margin=dict(l=80, r=80, t=100, b=80)
)

fig_forecast.show()

Os dados permitiram se construir um modelo com acurácia de 87.65% (MAPE 12.35%), aceitável para forecasting operacional. O erro absoluto médio por mês ficou em USD 8,051, o que é razoável considerando que suas vendas oscilam entre USD 15 mil e USD 120 mil. A previsão para 2018 projeta vendas entre USD 35 mil e USD 72 mil mensais, seguindo o padrão sazonal histórico com maior incerteza nos meses mais distantes.

## 🔮 Modelagem 2: Segmentação de Clientes (Análise RFM & K-Means)

**Objetivo:** Transcender a visão puramente transacional e segmentar a base de clientes por comportamento, identificando os perfis que sustentam a lucratividade versus aqueles que pressionam as margens através de descontos.

**O Modelo (RFM + K-Means)**

Utilizamos a metodologia RFM (Recência, Frequência e Valor Monetário) para criar uma "impressão digital" de cada cliente. Esta abordagem transforma dados brutos em três dimensões comportamentais: Recência (o quão recente foi sua compra), Frequência (o quanto ele é leal) e Monetário (o valor total investido).

Para agrupar esses comportamentos, aplicamos o algoritmo K-Means, um modelo de aprendizado não supervisionado que identifica naturalmente os padrões de consumo sem necessidade de rótulos prévios, separando a base em clusters distintos de valor estratégico.

## Valor de Negócio (Insights Acionáveis)

A clusterização não é um fim, mas um meio para a personalização estratégica:

* **Identificação de "Discount Junkies":** Clientes que apenas convertem com descontos elevados (prejudicando a margem).

* **Segmento VIP (High Value):** Clientes de alto ticket e baixa sensibilidade a preço, que devem receber foco em retenção e não em promoções agressivas.

* **Oportunidades de Reativação:** Clientes de alto valor histórico com longa inatividade, alvo prioritário para campanhas de win-back.

In [85]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Preparação dos dados RFM
# Supondo que 'df' seja seu dataset de vendas
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (reference_date - x.max()).days, # Recency
    'Order ID': 'count',                                    # Frequency
    'Sales': 'sum'                                          # Monetary
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

# 2. Normalização
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# 3. K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Resultado: verifique a média de cada grupo
print(rfm.groupby('Cluster').mean())

import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Cálculo RFM ---
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (reference_date - x.max()).days,
    'Order ID': 'count',
    'Sales': 'sum'
}).rename(columns={'Order Date': 'Recency', 'Order ID': 'Frequency', 'Sales': 'Monetary'})

# --- Clusterização ---
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)
kmeans = KMeans(n_clusters=3, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled).astype(str)

# --- Visualização: Scatter 3D ---
fig = px.scatter_3d(rfm, x='Recency', y='Frequency', z='Monetary', 
                    color='Cluster', title="Segmentação de Clientes (RFM)",
                    labels={'Cluster': 'Grupo de Cliente'})
fig.show()

            Recency  Frequency     Monetary
Cluster                                    
0         82.695560  10.627907  1968.827812
1        518.779661   7.669492  1565.211752
2         83.544554  20.108911  5847.773854


## 🔮 Modelagem 3: Classificação de Propensão (Previsão de Comportamento)

**Objetivo:** Antecipar a probabilidade de um cliente exigir descontos agressivos antes mesmo da conclusão da venda, permitindo uma negociação proativa e protegida.

**O Modelo de Classificação**

Trata-se de um problema de Aprendizado Supervisionado. Treinamos um algoritmo de classificação (Random Forest/Decision Tree) para aprender os padrões associados a pedidos com descontos acima de 30% (nossa "Zona de Risco"). O modelo utiliza variáveis como Categoria de Produto, Região, Histórico de Compras e Tipo de Envio para calcular a probabilidade de uma venda cair na margem de prejuízo.

**Metodologia de Validação e Métricas**

Para assegurar que o modelo não apenas "chute", mas seja um ativo operacional confiável para a equipe de vendas, utilizamos:

* **Matriz de Confusão:** Para visualizar a capacidade do modelo em identificar corretamente os "pedidos de risco" (verdadeiros positivos) e evitar bloqueios desnecessários em vendas saudáveis (falsos positivos).

* **Precision e Recall:** Métricas cruciais para este contexto. Priorizamos o Recall para garantir que capturemos a maior parte das vendas de risco, e a Precision para garantir que a equipe de vendas tenha confiança nos alertas gerados pelo modelo.

In [84]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Definindo o Target (Ex: Pedido com desconto > 20%)
df['High_Discount'] = (df['Discount'] > 0.2).astype(int)

# 2. Seleção de features (exemplo)
features = ['Sales', 'Quantity', 'Profit', 'Category_Encoded'] # Ajuste as colunas
X = df[features]
y = df['High_Discount']

# 3. Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# 4. Avaliação
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


import plotly.express as px
from sklearn.ensemble import RandomForestClassifier

# --- Treino ---
df['High_Discount'] = (df['Discount'] > 0.2).astype(int)
features = ['Sales', 'Quantity', 'Profit'] 
X = df[features]
y = df['High_Discount']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# --- Visualização: Importância das Variáveis ---
feature_imp = pd.DataFrame({'Feature': features, 'Importância': model.feature_importances_})
fig_imp = px.bar(feature_imp, x='Importância', y='Feature', orientation='h',
                 title="O que mais influencia o risco de descontos?",
                 color='Importância', color_continuous_scale='Viridis')
fig_imp.show()

KeyError: "['Category_Encoded'] not in index"

## 🚀 Análise Prescritiva: Otimização e "What-If" (Simulação de Cenários)

**Objetivo:** Definir as regras de negócio otimizadas que maximizam o lucro líquido, cruzando os insights dos modelos de ML com a realidade operacional da Superstore.

**Simulação "What-If"**

A etapa de otimização consiste em modelar diferentes "fronteiras de desconto". Através de simulações computacionais, alteramos a política de descontos da empresa para observar o efeito cascata no lucro global. Respondemos à pergunta: "Se restringirmos o teto de descontos em X% nas categorias inelásticas identificadas, quanto lucro recuperaremos mantendo a estabilidade da demanda?"

**A Nova Política de Preços**

Esta etapa encerra o ciclo de análise, entregando um guia de execução para a diretoria:

* **Definição de "Teto de Desconto" por Categoria:** Regras claras baseadas na elasticidade real (e não na percepção intuitiva).

* **Mitigação de Margem:** Substituição da estratégia de "volume a qualquer custo" pela estratégia de "volume rentável".

* **Conclusão Estratégica:** O plano definitivo que conecta a previsão de vendas (Forecasting) com a realidade comportamental da base (Clusters e Classificação), estabelecendo o caminho para o crescimento sustentável da margem operacional.

In [83]:
def simulate_profit(df, max_discount):
    # Copia o dataframe para não alterar o original
    df_sim = df.copy()
    
    # Aplica a regra de negócio: Se desconto > max, limita ao max
    df_sim['New_Discount'] = df_sim['Discount'].apply(lambda x: min(x, max_discount))
    
    # Recalcula o lucro aproximado (simplificado)
    # Lucro = Vendas * (Margem original) - Impacto do Desconto
    # Isso é uma aproximação para cenário de simulação
    df_sim['Simulated_Profit'] = df_sim['Sales'] * (1 - df_sim['New_Discount']) 
    
    return df_sim['Simulated_Profit'].sum()

# Comparação
current_profit = df['Profit'].sum()
simulated_profit = simulate_profit(df, max_discount=0.20)

print(f"Lucro Atual: R$ {current_profit:,.2f}")
print(f"Lucro com Teto de 20% de Desconto: R$ {simulated_profit:,.2f}")
print(f"Ganho Potencial: R$ {simulated_profit - current_profit:,.2f}")


import plotly.graph_objects as go

# --- Cálculo ---
lucro_atual = df['Profit'].sum()
# Simulação simples: teto de 20%
teto = 0.20
df_sim = df.copy()
df_sim['New_Discount'] = df_sim['Discount'].apply(lambda x: min(x, teto))
lucro_otimizado = (df_sim['Sales'] * (1 - df_sim['New_Discount'])).sum()
ganho = lucro_otimizado - lucro_atual

# --- Visualização: Gráfico Waterfall ---
fig_waterfall = go.Figure(go.Waterfall(
    name="Lucro", orientation="v",
    measure=["absolute", "relative", "total"],
    x=["Lucro Atual", "Ganho por Otimização", "Lucro Projetado"],
    y=[lucro_atual, ganho, lucro_otimizado],
    connector={"line": {"color": "rgb(63, 63, 63)"}},
))
fig_waterfall.update_layout(title="Impacto Financeiro da Otimização de Descontos")
fig_waterfall.show()

Lucro Atual: R$ 286,397.02
Lucro com Teto de 20% de Desconto: R$ 2,062,157.24
Ganho Potencial: R$ 1,775,760.22
